In [3]:
# #install all these below
# !pip install langchain langchain-community langchain-huggingface
# !pip install torch
# !pip install transformers
# !pip install faiss-cpu
# !pip install gemma
# !pip install groq
# !pip install sentence-transformers
# !pip install datasets
# !pip install rouge-score
# !pip install bert-score
# !pip install langchain_groq
# !pip install pyarrow
# !pip install transformers accelerate
# !pip install tqdm
#!pip install accelerate bitsandbytes
#!pip install tf-keras




In [59]:
import pandas as pd
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_groq import ChatGroq  
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from sklearn.metrics.pairwise import cosine_similarity
import torch
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import warnings
import pickle
import json
from datetime import datetime

from datetime import datetime
warnings.filterwarnings('ignore')
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline

from langchain.chains import RetrievalQA
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
pd.set_option("display.max_colwidth", None)

In [34]:
   # Replace 'your_hugging_face_token' with the token you copied
os.environ['HF_TOKEN'] = 'Enter_your_key'

from huggingface_hub import login

login(token=os.environ['HF_TOKEN'])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



Using device: cpu


In [24]:
import pandas as pd

# Load passages (knowledge base)
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

# Load test data (questions and answers for evaluation)
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Inspect the data
print("Passages DataFrame (first 2 rows):")
print(passages_df.head(2))
print(f"Shape: {passages_df.shape}")
print("\nColumns: ", passages_df.columns.tolist())

print("\nTest DataFrame (first 2 rows):")
print(test_df.head(2))
print(f"Shape: {test_df.shape}")
print("\nColumns: ", test_df.columns.tolist())

Passages DataFrame (first 2 rows):
                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
Shape: (40221, 1)

Columns:  ['passage']

Test DataFrame (first 2 rows):
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  are: epidermal growt...   

                                 relevant_passage_ids  
id                                                     
0   [20598273, 6650562, 15829955, 15617541, 230011...  
1   [23821377, 24323361, 23382875, 222

In [21]:
passages_df.isnull().sum()

passage    0
dtype: int64

In [32]:
# Duplicate Analysis for BioASQ Dataset
# Check duplicates in passages_df and their impact on test_df

# 1. Check for duplicate passages
print("1. ANALYZING DUPLICATE PASSAGES")
print("-" * 40)

# Check exact duplicates based on passage text
duplicate_mask = passages_df.duplicated(subset=['passage'], keep=False)
duplicate_passages = passages_df[duplicate_mask]
unique_duplicate_texts = passages_df[passages_df.duplicated(subset=['passage'], keep='first')]

print(f"Total passages: {len(passages_df)}")
print(f"Duplicate passages (including all copies): {duplicate_mask.sum()}")
print(f"Unique duplicate texts (count of distinct duplicated passages): {len(unique_duplicate_texts)}")
print(f"Duplicate rate: {(duplicate_mask.sum() / len(passages_df)) * 100:.2f}%")


# 2. Check impact on test data
print(f"\n2. IMPACT ON TEST DATA")
print("-" * 40)

# Parse relevant_passage_ids from test_df
print("Parsing relevant passage IDs from test data...")

def parse_passage_ids(ids_str):
    """Parse the passage IDs string into a list of integers"""
    try:
        # Handle different formats like "[1, 2, 3]" or "1,2,3" etc.
        ids_str = str(ids_str).strip()
        if ids_str.startswith('[') and ids_str.endswith(']'):
            ids_str = ids_str[1:-1]
        
        # Split by comma and convert to integers
        ids = [int(x.strip().strip("'\"")) for x in ids_str.split(',') if x.strip()]
        return ids
    except:
        return []

# Apply parsing to get all referenced passage IDs
test_df['parsed_passage_ids'] = test_df['relevant_passage_ids'].apply(parse_passage_ids)

# Get all unique passage IDs referenced in test data
all_referenced_ids = set()
for ids_list in test_df['parsed_passage_ids']:
    all_referenced_ids.update(ids_list)

print(f"Total unique passage IDs referenced in test data: {len(all_referenced_ids)}")

# Check how many referenced passages are duplicates
referenced_duplicate_ids = set(duplicate_passages.index) & all_referenced_ids
print(f"Referenced passage IDs that are duplicates: {len(referenced_duplicate_ids)}")

if len(referenced_duplicate_ids) > 0:
    print(f"Percentage of referenced passages that are duplicates: {(len(referenced_duplicate_ids) / len(all_referenced_ids)) * 100:.2f}%")

# 3. Detailed impact analysis
print(f"\n3. DETAILED IMPACT ANALYSIS")
print("-" * 40)

# Check if removing duplicates would affect test questions
questions_affected = 0
total_question_passage_refs = 0

for idx, row in test_df.iterrows():
    passage_ids = row['parsed_passage_ids']
    total_question_passage_refs += len(passage_ids)
    
    # Check if any referenced passages are duplicates
    if any(pid in referenced_duplicate_ids for pid in passage_ids):
        questions_affected += 1

print(f"Total test questions: {len(test_df)}")
print(f"Questions referencing duplicate passages: {questions_affected}")
print(f"Percentage of questions affected: {(questions_affected / len(test_df)) * 100:.2f}%")
print(f"Total passage references in test data: {total_question_passage_refs}")

# 4. Recommendation
print(f"\n4. RECOMMENDATION")
print("-" * 40)

if duplicate_mask.sum() > 0:
    print("📊 DUPLICATE SUMMARY:")
    print(f"   • {duplicate_mask.sum()} duplicate passages found ({(duplicate_mask.sum() / len(passages_df)) * 100:.2f}% of dataset)")
    print(f"   • {questions_affected} test questions reference duplicate passages ({(questions_affected / len(test_df)) * 100:.2f}%)")
    
#     # if questions_affected > 0:
#     #     print("\n⚠️  WARNING: Removing duplicates will affect test questions!")
#     #     print("   Options:")
#     #     print("   1. Keep duplicates to maintain test data integrity")
#     #     print("   2. Remove duplicates but update test data references") 
#     #     print("   3. Remove duplicates and exclude affected test questions")
        
#         # Check if we can map duplicate IDs to kept IDs
#         print(f"\n💡 MAPPING ANALYSIS:")
#         duplicate_to_keep_mapping = {}
#         for passage_text in unique_duplicate_texts['passage']:
#             all_ids_with_text = passages_df[passages_df['passage'] == passage_text].index.tolist()
#             keep_id = min(all_ids_with_text)  # Keep the first occurrence
#             for dup_id in all_ids_with_text:
#                 if dup_id != keep_id:
#                     duplicate_to_keep_mapping[dup_id] = keep_id
        
#         print(f"   • Can map {len(duplicate_to_keep_mapping)} duplicate IDs to kept IDs")
#         print(f"   • This would preserve all test question references")
        
#     else:
#         print("\n✅ SAFE TO REMOVE: No test questions reference duplicate passages")
#         print("   Recommendation: Remove duplicates to improve efficiency")

else:
    print("✅ NO DUPLICATES FOUND: Dataset is clean")

print(f"\n" + "="*60)

DUPLICATE ANALYSIS FOR BIOASQ DATASET
1. ANALYZING DUPLICATE PASSAGES
----------------------------------------
Total passages: 40221
Duplicate passages (including all copies): 12252
Unique duplicate texts (count of distinct duplicated passages): 12246
Duplicate rate: 30.46%

2. IMPACT ON TEST DATA
----------------------------------------
Parsing relevant passage IDs from test data...
Total unique passage IDs referenced in test data: 40221
Referenced passage IDs that are duplicates: 12252
Percentage of referenced passages that are duplicates: 30.46%

3. DETAILED IMPACT ANALYSIS
----------------------------------------
Total test questions: 4719
Questions referencing duplicate passages: 3551
Percentage of questions affected: 75.25%
Total passage references in test data: 42608

4. RECOMMENDATION
----------------------------------------
📊 DUPLICATE SUMMARY:
   • 12252 duplicate passages found (30.46% of dataset)
   • 3551 test questions reference duplicate passages (75.25%)

⚠️  WARNING: R

In [31]:
# Quick look at duplicate passages

# Get duplicates
duplicates = passages_df[passages_df.duplicated(subset=['passage'], keep=False)]

# Show a sample duplicate group
sample_passage_text = duplicates['passage'].iloc[0]
sample_group = passages_df[passages_df['passage'] == sample_passage_text]

print(f"Sample duplicate passage text: '{sample_passage_text}'")
print(f"This passage appears {len(sample_group)} times with IDs:")
print(sample_group.index.tolist()[:20])  # Show first 20 IDs
print(f"Total occurrences: {len(sample_group)}")

print(f"\nOther duplicate examples:")
unique_duplicate_texts = duplicates['passage'].unique()[:5]
for i, text in enumerate(unique_duplicate_texts):
    count = (passages_df['passage'] == text).sum()
    print(f"{i+1}. Text: '{text}' appears {count} times")

Sample duplicate passage text: 'nan'
This passage appears 12220 times with IDs:
[97949, 98518, 100785, 117628, 125891, 227209, 227871, 261962, 355046, 367436, 367540, 378740, 489960, 497808, 508371, 509687, 619228, 619442, 623681, 631693]
Total occurrences: 12220

Other duplicate examples:
1. Text: 'nan' appears 12220 times
2. Text: '1. ' appears 24 times
3. Text: 'INTRODUCTION: About 3% of people will be diagnosed with epilepsy during their 
lifetime, but about 70% of people with epilepsy eventually go into remission.
METHODS AND OUTCOMES: We conducted a systematic review and aimed to answer the 
following clinical questions: What are the effects of starting antiepileptic 
drug treatment following a single seizure? What are the effects of drug 
monotherapy in people with partial epilepsy? What are the effects of additional 
drug treatments in people with drug-resistant partial epilepsy? What is the risk 
of relapse in people in remission when withdrawing antiepileptic drugs? What are 

In [26]:
# Duplicate Handling and Mapping Code

# Step 1: Identify duplicates and create mapping
print("Step 1: Creating duplicate mapping...")

# Find all duplicates based on passage content
duplicate_mask = passages_df.duplicated(subset=['passage'], keep=False)
duplicates_df = passages_df[duplicate_mask].copy()

print(f"Found {len(duplicates_df)} duplicate passage entries")

# Create mapping: duplicate_id -> keep_id (first occurrence)
duplicate_mapping = {}
keep_ids = []
remove_ids = []

# Group by passage content to handle duplicates
for passage_content, group in duplicates_df.groupby('passage'):
    group_ids = group.index.tolist()
    keep_id = min(group_ids)  # Keep the first occurrence (lowest ID)
    
    keep_ids.append(keep_id)
    
    # Map all other IDs to the keep_id
    for duplicate_id in group_ids:
        if duplicate_id != keep_id:
            duplicate_mapping[duplicate_id] = keep_id
            remove_ids.append(duplicate_id)

print(f"Will keep {len(keep_ids)} unique passages")
print(f"Will remove {len(remove_ids)} duplicate passages")
print(f"Created mapping for {len(duplicate_mapping)} duplicate IDs")

# Step 2: Create separate dataframes
print(f"\nStep 2: Creating separate dataframes...")

# Dataframe of duplicates to be removed
removed_duplicates_df = passages_df.loc[remove_ids].copy()
print(f"Removed duplicates dataframe: {len(removed_duplicates_df)} rows")

# Dataframe of kept duplicates (for reference)
kept_duplicates_df = passages_df.loc[keep_ids].copy()
print(f"Kept duplicates dataframe: {len(kept_duplicates_df)} rows")

# Clean passages dataframe (no duplicates)
clean_passages_df = passages_df.drop(remove_ids).copy()
print(f"Clean passages dataframe: {len(clean_passages_df)} rows")

# Verification
print(f"\nVerification:")
print(f"Original passages: {len(passages_df)}")
print(f"Clean passages: {len(clean_passages_df)}")
print(f"Removed duplicates: {len(removed_duplicates_df)}")
print(f"Total: {len(clean_passages_df) + len(removed_duplicates_df)}")

# Step 3: Update test data references safely
print(f"\nStep 3: Updating test data references...")

def update_passage_ids(ids_str, mapping):
    """Safely update passage IDs using the mapping"""
    try:
        # Parse the passage IDs string
        ids_str = str(ids_str).strip()
        if ids_str.startswith('[') and ids_str.endswith(']'):
            ids_str = ids_str[1:-1]
        
        # Extract individual IDs
        original_ids = [int(x.strip().strip("'\"")) for x in ids_str.split(',') if x.strip()]
        
        # Update IDs using mapping
        updated_ids = []
        for pid in original_ids:
            # Use mapped ID if it exists, otherwise keep original
            mapped_id = mapping.get(pid, pid)
            updated_ids.append(mapped_id)
        
        # Return as string in same format
        return str(updated_ids)
        
    except Exception as e:
        print(f"Error updating IDs for: {ids_str} - {e}")
        return ids_str  # Return original if parsing fails

# Create updated test dataframe
updated_test_df = test_df.copy()
updated_test_df['original_passage_ids'] = test_df['relevant_passage_ids']  # Keep backup
updated_test_df['relevant_passage_ids'] = test_df['relevant_passage_ids'].apply(
    lambda x: update_passage_ids(x, duplicate_mapping)
)

# Step 4: Verification of updates
print(f"\nStep 4: Verifying updates...")

# Parse updated IDs to verify all references are valid
def parse_ids_safe(ids_str):
    """Safely parse IDs for verification"""
    try:
        ids_str = str(ids_str).strip()
        if ids_str.startswith('[') and ids_str.endswith(']'):
            ids_str = ids_str[1:-1]
        return [int(x.strip().strip("'\"")) for x in ids_str.split(',') if x.strip()]
    except:
        return []

# Check all referenced IDs are in clean dataset
all_clean_ids = set(clean_passages_df.index)
all_referenced_ids = set()

for ids_str in updated_test_df['relevant_passage_ids']:
    ids = parse_ids_safe(ids_str)
    all_referenced_ids.update(ids)

missing_references = all_referenced_ids - all_clean_ids
valid_references = all_referenced_ids & all_clean_ids

print(f"Total unique IDs referenced in updated test data: {len(all_referenced_ids)}")
print(f"Valid references (exist in clean data): {len(valid_references)}")
print(f"Missing references: {len(missing_references)}")

if len(missing_references) > 0:
    print(f"⚠️  Warning: {len(missing_references)} referenced IDs not found in clean data")
    print(f"Missing IDs sample: {list(missing_references)[:10]}")
else:
    print("All test data references are valid!")


# Check that no new content was created
original_unique_passages = set(passages_df['passage'].dropna())
clean_unique_passages = set(clean_passages_df['passage'].dropna())
removed_unique_passages = set(removed_duplicates_df['passage'].dropna())

print(f"Original unique passage contents: {len(original_unique_passages)}")
print(f"Clean dataset unique contents: {len(clean_unique_passages)}")
print(f"Removed duplicates unique contents: {len(removed_unique_passages)}")

# Verify no content was lost or added
content_difference = original_unique_passages - clean_unique_passages
if len(content_difference) == 0:
    print(" All original content preserved in clean dataset")
else:
    print(f"Content difference found: {len(content_difference)} unique contents")

# Step 6: Show sample of updates
print(f"\nStep 6: Sample of updates...")
print("Sample original vs updated test references:")
for i in range(min(3, len(updated_test_df))):
    original = updated_test_df.iloc[i]['original_passage_ids']
    updated = updated_test_df.iloc[i]['relevant_passage_ids']
    if original != updated:
        print(f"Question {i}:")
        print(f"  Original: {original}")
        print(f"  Updated:  {updated}")
        break

# Step 7: Summary statistics
print(f"\nStep 7: Final Summary...")
print(f"DEDUPLICATION RESULTS:")
print(f"   • Original passages: {len(passages_df):,}")
print(f"   • Clean passages: {len(clean_passages_df):,}")
print(f"   • Removed duplicates: {len(removed_duplicates_df):,}")
print(f"   • Space saved: {len(removed_duplicates_df)/len(passages_df)*100:.1f}%")
print(f"   • Test questions: {len(updated_test_df):,}")
print(f"   • All references valid: {'Yes' if len(missing_references)==0 else 'No'}")

print(f"   Use: clean_passages_df and updated_test_df")

SAFE DUPLICATE REMOVAL AND MAPPING
Step 1: Creating duplicate mapping...
Found 12252 duplicate passage entries
Will keep 6 unique passages
Will remove 12246 duplicate passages
Created mapping for 12246 duplicate IDs

Step 2: Creating separate dataframes...
Removed duplicates dataframe: 12246 rows
Kept duplicates dataframe: 6 rows
Clean passages dataframe: 27975 rows

Verification:
Original passages: 40221
Clean passages: 27975
Removed duplicates: 12246
Total: 40221

Step 3: Updating test data references...

Step 4: Verifying updates...
Total unique IDs referenced in updated test data: 27975
Valid references (exist in clean data): 27975
Missing references: 0
✅ All test data references are valid!

Step 5: Data Leakage Verification...
Original unique passage contents: 27975
Clean dataset unique contents: 27975
Removed duplicates unique contents: 6
✅ NO DATA LEAKAGE: All original content preserved in clean dataset

Step 6: Sample of updates...
Sample original vs updated test references:
Qu

In [29]:
print(clean_passages_df.info())
print(updated_test_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 27975 entries, 9797 to 34894461
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   passage  27975 non-null  string
dtypes: string(1)
memory usage: 437.1 KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 4719 entries, 0 to 4718
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   question              4719 non-null   string
 1   answer                4719 non-null   string
 2   relevant_passage_ids  4719 non-null   object
 3   parsed_passage_ids    4719 non-null   object
 4   original_passage_ids  4719 non-null   string
dtypes: object(2), string(3)
memory usage: 221.2+ KB
None


In [47]:
updated_test_df.head()

,question,answer,relevant_passage_ids,parsed_passage_ids,original_passage_ids
id,,,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 97949, 97949, 15617541, 23001136, 9...","[20598273, 6650562, 15829955, 15617541, 230011...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[97949, 24323361, 23382875, 97949, 23787814, 9...","[23821377, 24323361, 23382875, 22247333, 23787...","[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004...","[21784067, 19297413, 15094122, 7515725, 332004...","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 97949, 22707570, 22955988, 24285305...","[22955974, 21622663, 22707570, 22955988, 24285...","[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 97949, 97949, 97949, 24265865, 9794...","[22867712, 23827649, 21618594, 23835909, 24265...","[22867712, 23827649, 21618594, 23835909, 24265..."


# Query Rewriting using Gemma

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Setup query rewriting model - using Gemma as per assignment requirements
model_name = "google/gemma-2-2b-it"  # Smallest Gemma model for CPU
query_tokenizer = AutoTokenizer.from_pretrained(model_name)
query_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float32
)


In [40]:
# Step 2: Query Rewriting Model Setup (CPU Optimized)

# Create query rewriting pipeline (remove device argument due to accelerate)
query_rewriter = pipeline(
    "text-generation",
    model=query_model,
    tokenizer=query_tokenizer,
    max_length=100
)

# Query rewriting function
def rewrite_query(original_query):
    """Rewrite query to improve retrieval using medical context"""
    prompt = f"Expand this medical question with more clinical terms: {original_query}\nExpanded question:"
    
    # Generate with proper parameters for Gemma
    result = query_rewriter(
        prompt, 
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        pad_token_id=query_tokenizer.eos_token_id,
        truncation=True
    )
    
    # Extract just the generated text after the prompt
    generated_text = result[0]['generated_text']
    rewritten = generated_text.split("Expanded question:")[-1].strip()
    
    # Extract only the question part (before any explanation)
    if '"' in rewritten:
        rewritten = rewritten.split('"')[1] if rewritten.count('"') >= 2 else rewritten
    elif '?' in rewritten:
        rewritten = rewritten.split('?')[0] + '?'
    
    # Fallback to original if rewriting fails
    return rewritten if rewritten and len(rewritten) > 10 else original_query



Device set to use cpu


In [41]:
# Test query rewriting
sample_query = test_df['question'].iloc[0]
rewritten_query = rewrite_query(sample_query)
print(f"Original: {sample_query}")
print(f"Rewritten: {rewritten_query}")

Original: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Rewritten: **Considering the observed familial aggregation of Hirschsprung disease, what is the relative contribution of genetic and environmental factors to the pathogenesis of this condition, including potential candidate genes and their functional roles in the development of this disorder?


# Embeddings and FAISS Vector Database Setup

In [42]:

# Load embedding model for passages
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # Fast, efficient embedder

# Prepare passages (using clean deduplicated data)
passages_list = clean_passages_df['passage'].fillna('').tolist()
passage_ids = clean_passages_df.index.tolist()

# Create embeddings for all passages
print("Creating embeddings...")
embeddings = embedder.encode(passages_list, show_progress_bar=True)





modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/875 [00:00<?, ?it/s]

In [43]:
# Create FAISS index
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner product for similarity
faiss_index.add(embeddings.astype(np.float32))

print(f"FAISS index created with {faiss_index.ntotal} passages")

# Retrieval function
def retrieve_passages(query, top_k=10):
    """Retrieve top-k most similar passages for a query"""
    # Rewrite query using Gemma
    rewritten = rewrite_query(query)
    
    # Embed the rewritten query
    query_embedding = embedder.encode([rewritten])
    
    # Search FAISS
    scores, indices = faiss_index.search(query_embedding.astype(np.float32), top_k)
    
    # Get passage texts and IDs
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        passage_id = passage_ids[idx]
        passage_text = passages_list[idx]
        results.append({
            'rank': i+1,
            'passage_id': passage_id,
            'passage': passage_text,
            'score': float(score)
        })
    
    return rewritten, results

FAISS index created with 27975 passages


In [46]:
# Test retrieval
test_query = updated_test_df['question'].iloc[0]
rewritten_query, retrieved_passages = retrieve_passages(test_query, top_k=5)

print(f"Original query: {test_query}")
print(f"Rewritten query: {rewritten_query}")
print(f"Top 3 retrieved passages:")
for i, result in enumerate(retrieved_passages[:3]):
    print(f"{i+1}. Score: {result['score']:.3f}")
    print(f"   Text: {result['passage'][:100]}...")

Original query: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Rewritten query: What is the genetic basis underlying Hirschsprung disease (HD), and how significant is the contribution of Mendelian and multifactorial inheritance to its development?
Top 3 retrieved passages:
1. Score: 0.760
   Text: Hirschsprung's disease (HSCR) is a fairly frequent cause of intestinal 
obstruction in children. It ...
2. Score: 0.722
   Text: Hirschsprung disease (HSCR, aganglionic megacolon) represents the main genetic 
cause of functional ...
3. Score: 0.685
   Text: Hirschsprung's disease is characterized by the absence of ganglion cells in the 
myenteric and submu...


# Evaluation- Gemma + Faiss system

In [51]:
# Step 4: Evaluation Metrics for RAG System
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import numpy as np

# Initialize ROUGE scorer
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def evaluate_retrieval_metrics(test_sample_size=50):
    """Evaluate RAG system with MAP, MRR, ROUGE-L, and BERT-F1"""
    
    # Use subset for faster evaluation
    test_subset = updated_test_df.head(test_sample_size)
    
    all_retrieved_passages = []
    all_reference_answers = []
    retrieval_ranks = []
    
    print(f"Evaluating on {len(test_subset)} questions...")
    
    for idx, row in test_subset.iterrows():
        question = row['question'] 
        reference_answer = row['answer']
        relevant_passage_ids = eval(row['relevant_passage_ids'])  # Use updated mapped IDs
        
        # Get retrieved passages
        rewritten_query, retrieved_results = retrieve_passages(question, top_k=10)
        retrieved_passage_ids = [r['passage_id'] for r in retrieved_results]
        
        # Combine retrieved passage texts for evaluation
        retrieved_text = " ".join([r['passage'] for r in retrieved_results[:3]])  # Top 3 passages
        
        all_retrieved_passages.append(retrieved_text)
        all_reference_answers.append(reference_answer)
        
        # Calculate retrieval metrics (MAP/MRR)
        relevant_ranks = []
        for rel_id in relevant_passage_ids:
            if rel_id in retrieved_passage_ids:
                rank = retrieved_passage_ids.index(rel_id) + 1
                relevant_ranks.append(rank)
        
        retrieval_ranks.append(relevant_ranks)
        
        if idx % 20 == 0:
            print(f"Processed {idx+1}/{len(test_subset)} questions")
    
    # Calculate MAP (Mean Average Precision)
    def calculate_map(retrieval_ranks):
        total_ap = 0
        for ranks in retrieval_ranks:
            if not ranks:  # No relevant documents retrieved
                ap = 0
            else:
                ap = np.mean([len([r for r in ranks if r <= rank]) / rank for rank in ranks])
            total_ap += ap
        return total_ap / len(retrieval_ranks)
    
    # Calculate MRR (Mean Reciprocal Rank)
    def calculate_mrr(retrieval_ranks):
        total_rr = 0
        for ranks in retrieval_ranks:
            rr = 1.0 / min(ranks) if ranks else 0
            total_rr += rr
        return total_rr / len(retrieval_ranks)
    
    # Calculate ROUGE-L scores
    rouge_scores = []
    for retrieved, reference in zip(all_retrieved_passages, all_reference_answers):
        score = rouge_scorer_obj.score(reference, retrieved)
        rouge_scores.append(score['rougeL'].fmeasure)
    
    # Calculate BERT-F1 scores
    print("Calculating BERT scores...")
    P, R, F1 = bert_score(all_retrieved_passages, all_reference_answers, lang='en')
    bert_f1_scores = F1.numpy()
    
    # Compile results
    map_score = calculate_map(retrieval_ranks)
    mrr_score = calculate_mrr(retrieval_ranks)
    avg_rouge_l = np.mean(rouge_scores)
    avg_bert_f1 = np.mean(bert_f1_scores)
    
    return {
        'MAP': map_score,
        'MRR': mrr_score, 
        'ROUGE-L': avg_rouge_l,
        'BERT-F1': avg_bert_f1,
        'sample_size': test_sample_size
    }



In [52]:
# Run evaluation
print("Starting evaluation...")
results = evaluate_retrieval_metrics(test_sample_size=50)  # Start with 50 for speed

print("\n" + "="*50)
print("RAG SYSTEM EVALUATION RESULTS")
print("="*50)
print(f"Sample Size: {results['sample_size']} questions")
print(f"MAP (Mean Average Precision): {results['MAP']:.4f}")
print(f"MRR (Mean Reciprocal Rank): {results['MRR']:.4f}")
print(f"ROUGE-L Score: {results['ROUGE-L']:.4f}")
print(f"BERT-F1 Score: {results['BERT-F1']:.4f}")
print("="*50)

Starting evaluation...
Evaluating on 50 questions...
Processed 1/50 questions
Processed 21/50 questions
Processed 41/50 questions
Calculating BERT scores...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



RAG SYSTEM EVALUATION RESULTS
Sample Size: 50 questions
MAP (Mean Average Precision): 0.7068
MRR (Mean Reciprocal Rank): 0.7862
ROUGE-L Score: 0.0589
BERT-F1 Score: 0.8135


Evaluation Results Analysis:
Retrieval Performance

MAP: 0.7068 - Very good! Your system finds relevant passages 70.7% of the time
MRR: 0.7862 - Excellent! Relevant passages appear in top positions ~78% of the time

Content Quality:

BERT-F1: 0.8135 - Outstanding! 81% semantic similarity between retrieved content and reference answers
ROUGE-L: 0.0589 - Low, but expected (retrieved passages vs. direct answers have different structures)



# LLM integration for full RAG pipeline

In [53]:
# Step 5: LLM Integration for Complete RAG Pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Setup answer generation LLM (using a lightweight model for CPU/GPU efficiency)
def setup_answer_generator():
    """Setup LLM for answer generation - using Llama as per assignment"""
    try:
        # Use Llama-2-7B-chat as specified in assignment requirements
        model_name = "meta-llama/Llama-2-7b-chat-hf"
        answer_tokenizer = AutoTokenizer.from_pretrained(model_name)
        answer_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else "cpu"
        )
        
        if answer_tokenizer.pad_token is None:
            answer_tokenizer.pad_token = answer_tokenizer.eos_token
            
        answer_generator = pipeline(
            "text-generation",
            model=answer_model,
            tokenizer=answer_tokenizer,
            max_length=200,
            truncation=True
        )
        
        print("✅ Llama-2-7B-chat loaded successfully!")
        return answer_generator
        
    except Exception as e:
        print(f"❌ Error loading Llama model: {e}")
        print("Trying fallback to Flan-T5...")
        
        try:
            # Fallback to Flan-T5 (also assignment-compliant)
            model_name = "google/flan-t5-base"
            answer_tokenizer = AutoTokenizer.from_pretrained(model_name)
            answer_model = AutoModelForCausalLM.from_pretrained(model_name)
            
            answer_generator = pipeline(
                "text2text-generation",
                model=answer_model,
                tokenizer=answer_tokenizer,
                max_length=200
            )
            
            print("✅ Flan-T5 loaded as fallback!")
            return answer_generator
            
        except Exception as e2:
            print(f"❌ Error loading fallback model: {e2}")
            return None



In [56]:
def complete_rag_pipeline(question, top_k=5):
    """Complete RAG: Query Rewriting + Retrieval + Answer Generation"""
    
    # Step 1: Query rewriting with Gemma
    rewritten_query = rewrite_query(question)
    
    # Step 2: Retrieve relevant passages
    _, retrieved_passages = retrieve_passages(question, top_k=top_k)
    
    # Step 3: Prepare context from retrieved passages
    context = "\n\n".join([
        f"Passage {i+1}: {passage['passage'][:300]}..."  # Limit passage length
        for i, passage in enumerate(retrieved_passages[:3])  # Use top 3 passages
    ])
    
    # Step 4: Generate answer using retrieved context
    prompt = f"""Based on the following medical information, answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""
    
    try:
        # Generate answer
        response = answer_generator(
            prompt,
            max_new_tokens=150,  # Increased from 100
            do_sample=True,
            temperature=0.7,
            pad_token_id=answer_generator.tokenizer.eos_token_id,
            truncation=True
        )
        
        # Extract generated answer
        generated_text = response[0]['generated_text']
        answer = generated_text.split("Answer:")[-1].strip()
        
        # Clean up answer (increased limit)
        if len(answer) > 500:  # Increased from 300
            answer = answer[:500] + "..."
            
    except Exception as e:
        print(f"Error generating answer: {e}")
        answer = "I couldn't generate an answer based on the retrieved information."
    
    return {
        'original_question': question,
        'rewritten_query': rewritten_query,
        'retrieved_passages': retrieved_passages,
        'generated_answer': answer,
        'context_used': context
    }

In [54]:

# Initialize the answer generator
print("Setting up answer generation model...")
answer_generator = setup_answer_generator()

if answer_generator:
    # Test the complete RAG pipeline
    print("\n" + "="*60)
    print("TESTING COMPLETE RAG PIPELINE")
    print("="*60)
    
    test_question = "What causes Hirschsprung disease?"
    
    print(f"Testing question: {test_question}")
    print("\nProcessing...")
    
    # Run complete RAG pipeline
    rag_result = complete_rag_pipeline(test_question)
    
    # Display results
    print(f"\n📝 Original Question: {rag_result['original_question']}")
    print(f"\n🔄 Rewritten Query: {rag_result['rewritten_query']}")
    print(f"\n📚 Top Retrieved Passages:")
    for i, passage in enumerate(rag_result['retrieved_passages'][:2]):
        print(f"   {i+1}. Score: {passage['score']:.3f}")
        print(f"      Text: {passage['passage'][:150]}...")
    
    print(f"\n💡 Generated Answer: {rag_result['generated_answer']}")
    
    print("\n✅ Complete RAG pipeline working successfully!")
    
else:
    print("❌ Could not initialize answer generator. Please check the model setup.")

Setting up answer generation model...


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Device set to use cpu


✅ Llama-2-7B-chat loaded successfully!

TESTING COMPLETE RAG PIPELINE
Testing question: What causes Hirschsprung disease?

Processing...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



📝 Original Question: What causes Hirschsprung disease?

🔄 Rewritten Query: What is the etiology and pathogenesis of Hirschsprung disease, including the role of the neural crest cells and the potential contribution of genetic and environmental factors?

📚 Top Retrieved Passages:
   1. Score: 0.720
      Text: Hirschsprung's disease (HSCR) is a fairly frequent cause of intestinal 
obstruction in children. It is characterized as a sex-linked heterogonous 
dis...
   2. Score: 0.709
      Text: Hirschsprung disease is a congenital form of aganglionic megacolon that results 
from cristopathy. Hirschsprung disease usually occurs as a sporadic d...

💡 Generated Answer: Based on the provided passages, Hirschsprung disease is caused by cristopathy, which is a congenital form of aganglionic megacolon.

✅ Complete RAG pipeline working successfully!


In [57]:
# Function to evaluate multiple questions
def evaluate_complete_rag(num_questions=5):
    """Test RAG pipeline on multiple questions"""
    if not answer_generator:
        print("Answer generator not available")
        return
        
    print(f"\n{'='*60}")
    print(f"EVALUATING COMPLETE RAG ON {num_questions} QUESTIONS")
    print(f"{'='*60}")
    
    for i in range(min(num_questions, len(updated_test_df))):
        question = updated_test_df.iloc[i]['question']
        reference_answer = updated_test_df.iloc[i]['answer']
        
        print(f"\n--- Question {i+1} ---")
        print(f"Q: {question}")
        
        # Get RAG result
        rag_result = complete_rag_pipeline(question)
        
        print(f"Generated: {rag_result['generated_answer']}")  # Show full generated answer
        print(f"Reference: {reference_answer}")  # Show full reference answer
        print(f"Retrieval Score: {rag_result['retrieved_passages'][0]['score']:.3f}")

# Uncomment to test on multiple questions
evaluate_complete_rag(3)


EVALUATING COMPLETE RAG ON 3 QUESTIONS

--- Question 1 ---
Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Generated: Hirschsprung disease is a mendelian disorder.
Reference: Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.
Retrieval Score: 0.727

--- Question 2 ---
Q: List signaling molecules (ligands) that interact with the receptor EGFR?
Generated: EGFR ligands that interact with the receptor EGFR are:

1. Epidermal growth factor (EGF)
2. Transforming growth factor-alpha (TGF-alpha)
3. Amphiregulin
4. Epiregulin
5. Heparin-binding EGF-like growth factor (HB-EGF)
Reference: The 7 known EGFR ligands  are: epide

# Langchain setup


In [66]:
# Step 1: Setup LangChain Components
print("Setting up LangChain RAG pipeline...")

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# Prepare documents for LangChain
print("Preparing documents...")
documents = []
for idx, row in clean_passages_df.iterrows():
    if pd.notna(row['passage']) and row['passage'].strip():
        doc = Document(
            page_content=row['passage'],
            metadata={"passage_id": idx, "source": "bioasq"}
        )
        documents.append(doc)

print(f"Created {len(documents)} documents")

# Create FAISS vector store
print("Creating FAISS vector store...")
vectorstore = FAISS.from_documents(documents, embeddings)

# Setup LLM pipeline properly for LangChain
if 'answer_generator' in globals() and answer_generator:
    # Create proper LangChain HuggingFace pipeline
    llm = HuggingFacePipeline(
        pipeline=answer_generator,
        model_kwargs={
            "max_new_tokens": 100,
            "do_sample": True, 
            "temperature": 0.7,
            "return_full_text": False,
            "truncation": True
        }
    )
    print("✅ LangChain HuggingFace pipeline ready")
else:
    print("❌ Please run Step 5 first to load the answer generator")

# Step 2: Out-of-Context Detection
def is_medical_question(question):
    """Simple out-of-context detection for medical questions"""
    medical_keywords = [
        'disease', 'disorder', 'syndrome', 'treatment', 'therapy', 'medicine', 
        'drug', 'medication', 'symptom', 'diagnosis', 'patient', 'clinical',
        'medical', 'health', 'cancer', 'tumor', 'infection', 'virus', 'bacteria',
        'gene', 'genetic', 'protein', 'enzyme', 'cell', 'tissue', 'organ',
        'blood', 'heart', 'brain', 'liver', 'kidney', 'lung', 'diabetes',
        'hypertension', 'covid', 'vaccine', 'antibody', 'immune', 'pathology'
    ]
    
    question_lower = question.lower()
    
    # Check for medical keywords
    medical_score = sum(1 for keyword in medical_keywords if keyword in question_lower)
    
    # Check for non-medical topics
    non_medical_keywords = [
        'economy', 'politics', 'sports', 'weather', 'food', 'recipe',
        'travel', 'music', 'movie', 'game', 'fashion', 'shopping',
        'tariff', 'election', 'football', 'basketball', 'restaurant'
    ]
    
    non_medical_score = sum(1 for keyword in non_medical_keywords if keyword in question_lower)
    
    # Simple scoring system
    if non_medical_score > 0:
        return False
    if medical_score > 0:
        return True
    
    # For ambiguous cases, check retrieval relevance
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    if retrieved_docs:
        avg_score = np.mean([doc.metadata.get('score', 0) for doc in retrieved_docs])
        return avg_score > 0.5  # Threshold for relevance
    
    return False

# Step 3: Query Rewriting with Context Filtering
def enhanced_query_rewriter(question):
    """Enhanced query rewriting with medical context checking"""
    
    # First check if question is medical
    if not is_medical_question(question):
        return None, "I can only answer medical and health-related questions."
    
    # Use Gemma for query rewriting (your existing function)
    try:
        rewritten = rewrite_query(question)
        return rewritten, None
    except Exception as e:
        print(f"Query rewriting failed: {e}")
        return question, None  # Fallback to original question

# Step 4: Custom RAG Chain with Filtering
class FilteredRetrievalQA:
    def __init__(self, vectorstore, llm, embeddings):
        self.vectorstore = vectorstore
        self.llm = llm
        self.embeddings = embeddings
        
        # Custom prompt template (shorter to avoid length issues)
        self.prompt_template = PromptTemplate(
            template="""Use this medical information to answer the question briefly.

Context: {context}

Question: {question}

Answer:""",
            input_variables=["context", "question"]
        )
    
    def run(self, question):
        """Run the filtered RAG pipeline"""
        
        # Step 1: Check if question is medical
        rewritten_query, error_msg = enhanced_query_rewriter(question)
        
        if error_msg:
            return {
                "result": error_msg,
                "source_documents": [],
                "rewritten_query": None,
                "context_check": "failed"
            }
        
        # Step 2: Retrieve relevant documents
        try:
            retrieved_docs = self.vectorstore.similarity_search(rewritten_query, k=5)
            
            if not retrieved_docs:
                return {
                    "result": "I couldn't find relevant medical information to answer your question.",
                    "source_documents": [],
                    "rewritten_query": rewritten_query,
                    "context_check": "no_docs"
                }
            
            # Step 3: Prepare context
            context = "\n\n".join([doc.page_content[:300] for doc in retrieved_docs[:3]])
            
            # Step 3: Prepare shorter context to avoid length issues
            context = "\n".join([doc.page_content[:200] for doc in retrieved_docs[:2]])
            
            # Step 4: Generate answer using LangChain
            prompt = self.prompt_template.format(context=context, question=question)
            
            # Use LangChain LLM properly
            response = self.llm(prompt)
            
            # LangChain returns string directly
            answer = response.strip() if isinstance(response, str) else str(response)
            
            # Clean up answer
            if "Answer:" in answer:
                answer = answer.split("Answer:")[-1].strip()
            
            if not answer or len(answer) < 10:
                answer = "Based on the retrieved medical information, I found relevant content but couldn't generate a clear answer."
            
            return {
                "result": answer,
                "source_documents": retrieved_docs,
                "rewritten_query": rewritten_query,
                "context_check": "passed"
            }
            
        except Exception as e:
            return {
                "result": f"I encountered an error processing your medical question: {str(e)}",
                "source_documents": [],
                "rewritten_query": rewritten_query,
                "context_check": "error"
            }


Setting up LangChain RAG pipeline...
Preparing documents...
Created 27975 documents
Creating FAISS vector store...
✅ LangChain HuggingFace pipeline ready


In [67]:
# Step 5: Initialize the Filtered RAG Chain
if 'llm' in globals():
    filtered_rag = FilteredRetrievalQA(vectorstore, llm, embeddings)
    print("LangChain RAG with filtering initialized!")
    
    # Test the system
    print("\n" + "="*60)
    print("TESTING LANGCHAIN RAG WITH OUT-OF-CONTEXT FILTERING")
    print("="*60)
    
    # Test medical question
    medical_q = "What causes Hirschsprung disease?"
    print(f"\n Medical Question: {medical_q}")
    result = filtered_rag.run(medical_q)
    print(f" Response: {result['result'][:200]}...")
    print(f" Rewritten Query: {result['rewritten_query']}")
    print(f" Context Check: {result['context_check']}")
    
    # Test non-medical question
    non_medical_q = "What is the effect of tariffs on the economy?"
    print(f"\n Non-Medical Question: {non_medical_q}")
    result = filtered_rag.run(non_medical_q)
    print(f" Response: {result['result']}")
    print(f" Context Check: {result['context_check']}")
    
else:
    print("LLM not available. Please run the previous steps first.")

✅ LangChain RAG with filtering initialized!

TESTING LANGCHAIN RAG WITH OUT-OF-CONTEXT FILTERING

🩺 Medical Question: What causes Hirschsprung disease?
✅ Response: Hirschsprung disease is caused by cristopathy, which is a failure of the nerve cells to migrate to the distal end of the colon during fetal development....
📝 Rewritten Query: What are the specific genetic and developmental mechanisms that contribute to the pathogenesis of Hirschsprung disease, including the role of the neural crest cells and their migration, as well as potential environmental factors that may play a part in its etiology?
🔍 Context Check: passed

💼 Non-Medical Question: What is the effect of tariffs on the economy?
🚫 Response: I can only answer medical and health-related questions.
🔍 Context Check: failed


In [69]:
# Chatbot Interface Function
def medical_chatbot(question):
    """Simple chatbot interface"""
    if 'filtered_rag' in globals():
        result = filtered_rag.run(question)
        return result
    else:
        return {"result": "Chatbot not initialized. Please run the setup first."}



In [70]:
# Test more questions
test_questions = [
    "What is diabetes?",  # Medical - should work
    "How to cook pasta?",  # Non-medical - should reject
    "What are the symptoms of COVID-19?"  # Medical - should work
]

for q in test_questions:
    result = medical_chatbot(q)
    print(f"Q: {q}")
    print(f"A: {result['result']}")         
    print(f"Status: {result['context_check']}")
    print(f"Answer length: {len(result['result'])} characters\n")

Q: What is diabetes?
A: Diabetes mellitus is a group of metabolic disorders characterized by high blood sugar levels, including type 1 diabetes, type 2 diabetes, and gestational diabetes.
Status: passed
Answer length: 163 characters

Q: How to cook pasta?
A: I can only answer medical and health-related questions.
Status: failed
Answer length: 55 characters

Q: What are the symptoms of COVID-19?
A: The article does not provide a comprehensive list of symptoms of COVID-19, as it focuses on the persistence of symptoms in individuals who have recovered from SARS-CoV-
Status: passed
Answer length: 168 characters



# Saving for deployment

In [75]:
# Comprehensive Save Strategy for RAG System Deployment
import pickle
import json
import os
from datetime import datetime

# Create save directory
save_dir = "rag_system_checkpoint"
os.makedirs(save_dir, exist_ok=True)

print("🔄 Saving complete RAG system for deployment...")

# =============================================================================
# 1. CRITICAL DATA (Must Save)
# =============================================================================

print("1️⃣ Saving critical processed data...")

# Clean processed data
clean_passages_df.to_parquet(f'{save_dir}/clean_passages_df.parquet')
updated_test_df.to_parquet(f'{save_dir}/updated_test_df.parquet')

# Deduplication mapping
with open(f'{save_dir}/duplicate_mapping.pkl', 'wb') as f:
    pickle.dump(duplicate_mapping, f)

print("✅ Critical data saved")

# =============================================================================
# 2. FAISS INDEX & SUPPORTING DATA (Time-Critical)
# =============================================================================

print("2️⃣ Saving FAISS index and supporting data...")

# FAISS index (takes long to rebuild)
faiss.write_index(faiss_index, f'{save_dir}/bioasq_faiss_index.index')

# Supporting lists for FAISS
with open(f'{save_dir}/passages_list.pkl', 'wb') as f:
    pickle.dump(passages_list, f)

with open(f'{save_dir}/passage_ids.pkl', 'wb') as f:
    pickle.dump(passage_ids, f)

# LangChain vectorstore metadata
if 'vectorstore' in globals():
    vectorstore.save_local(f'{save_dir}/langchain_vectorstore')

print("✅ FAISS and vectorstore saved")

# =============================================================================
# 3. EVALUATION RESULTS & METRICS
# =============================================================================

print("3️⃣ Saving evaluation results...")

evaluation_results = {
    'MAP': 0.5911,
    'MRR': 0.6489,
    'ROUGE-L': 0.0640,
    'BERT-F1': 0.8095,
    'evaluation_date': datetime.now().isoformat(),
    'sample_size': 100,
    'model_components': {
        'query_rewriter': 'google/gemma-2-2b-it',
        'embedder': 'sentence-transformers/all-MiniLM-L6-v2',
        'answer_generator': 'meta-llama/Llama-2-7b-chat-hf',
        'framework': 'LangChain'
    }
}

with open(f'{save_dir}/evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("✅ Evaluation results saved")

# =============================================================================
# 4. SYSTEM CONFIGURATION
# =============================================================================

print("4️⃣ Saving system configuration...")

system_config = {
    'models': {
        'query_rewriter': 'google/gemma-2-2b-it',
        'embedder': 'sentence-transformers/all-MiniLM-L6-v2', 
        'answer_generator': 'meta-llama/Llama-2-7b-chat-hf'
    },
    'parameters': {
        'max_new_tokens': 150,
        'temperature': 0.7,
        'top_k_retrieval': 5,
        'embedding_dimension': 384,
        'faiss_index_type': 'IndexFlatIP'
    },
    'data_stats': {
        'total_passages': len(clean_passages_df),
        'total_test_questions': len(updated_test_df),
        'duplicates_removed': len(duplicate_mapping)
    },
    'medical_keywords': [
        'disease', 'disorder', 'syndrome', 'treatment', 'therapy', 'medicine', 
        'drug', 'medication', 'symptom', 'diagnosis', 'patient', 'clinical',
        'medical', 'health', 'cancer', 'tumor', 'infection', 'virus', 'bacteria',
        'gene', 'genetic', 'protein', 'enzyme', 'cell', 'tissue', 'organ',
        'blood', 'heart', 'brain', 'liver', 'kidney', 'lung', 'diabetes',
        'hypertension', 'covid', 'vaccine', 'antibody', 'immune', 'pathology'
    ],
    'non_medical_keywords': [
        'economy', 'politics', 'sports', 'weather', 'food', 'recipe',
        'travel', 'music', 'movie', 'game', 'fashion', 'shopping',
        'tariff', 'election', 'football', 'basketball', 'restaurant',
        'cooking', 'pasta', 'pizza', 'hotel', 'vacation', 'concert',
        'stock', 'investment', 'entertainment', 'celebrity', 'social media'
    ]
}

with open(f'{save_dir}/system_config.json', 'w') as f:
    json.dump(system_config, f, indent=2)

print("✅ System configuration saved")

# =============================================================================
# 5. SAMPLE CONVERSATIONS (For Testing)
# =============================================================================

print("5️⃣ Saving sample conversations...")

sample_conversations = [
    {
        'question': 'What causes Hirschsprung disease?',
        'expected_behavior': 'medical_answer',
        'context_check': 'passed'
    },
    {
        'question': 'What is diabetes?', 
        'expected_behavior': 'medical_answer',
        'context_check': 'passed'
    },
    {
        'question': 'How to cook pasta?',
        'expected_behavior': 'rejection',
        'context_check': 'failed'
    },
    {
        'question': 'What are the symptoms of COVID-19?',
        'expected_behavior': 'medical_answer', 
        'context_check': 'passed'
    }
]

with open(f'{save_dir}/sample_conversations.json', 'w') as f:
    json.dump(sample_conversations, f, indent=2)

print("✅ Sample conversations saved")

# =============================================================================
# 6. REQUIREMENTS FILE
# =============================================================================

print("6️⃣ Creating requirements.txt...")

requirements_content = """# Core ML Libraries
torch>=1.11.0
transformers>=4.36.0
sentence-transformers>=2.2.2
accelerate>=0.20.0
bitsandbytes>=0.41.0

# Vector Database & Search
faiss-cpu>=1.7.4
# Use faiss-gpu>=1.7.4 if you have GPU

# LangChain Framework
langchain>=0.0.350
langchain-community>=0.0.10
langchain-huggingface>=0.0.1

# Data Processing
pandas>=1.5.0
numpy>=1.21.0
datasets>=2.0.0

# Evaluation Metrics
rouge-score>=0.1.2
bert-score>=0.3.13
nltk>=3.8.0

# Utilities
tqdm>=4.64.0
matplotlib>=3.5.0
seaborn>=0.11.0

# Optional: For better performance
# scipy>=1.9.0
# scikit-learn>=1.1.0

# System Requirements:
# Python>=3.8
# RAM: 8GB+ recommended for models
# Storage: 20GB+ for model cache
# GPU: Optional but recommended for faster inference
"""

with open(f'{save_dir}/requirements.txt', 'w') as f:
    f.write(requirements_content)

print("✅ Requirements.txt created")

# =============================================================================
# 7. INSTALLATION SCRIPT
# =============================================================================

print("7️⃣ Creating installation script...")

install_script = """#!/bin/bash
# Installation script for RAG Medical Chatbot

echo "🚀 Installing RAG Medical Chatbot System..."

# Create virtual environment (optional but recommended)
python -m venv rag_env
source rag_env/bin/activate  # On Windows: rag_env\\Scripts\\activate

# Upgrade pip
pip install --upgrade pip

# Install PyTorch (CPU version - change for GPU if needed)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Install other requirements
pip install -r requirements.txt

# Download NLTK data
python -c "import nltk; nltk.download('punkt')"

echo "✅ Installation complete!"
echo "📝 Next steps:"
echo "1. Activate environment: source rag_env/bin/activate"
echo "2. Run: python -c 'exec(open(\"load_system.py\").read())'"
echo "3. Your RAG system will be ready!"
"""

with open(f'{save_dir}/install.sh', 'w') as f:
    f.write(install_script)

# Make script executable
os.chmod(f'{save_dir}/install.sh', 0o755)

print("✅ Installation script created")

# =============================================================================
# 8. DEPLOYMENT SCRIPTS
# =============================================================================

print("8️⃣ Creating deployment scripts...")

# Quick load script
load_script = '''# Quick Load Script for RAG System
import pandas as pd
import pickle
import faiss
import json
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

def load_rag_system(save_dir="rag_system_checkpoint"):
    """Load complete RAG system from saved files"""
    print("Loading RAG system...")
    
    # Load data
    clean_passages_df = pd.read_parquet(f'{save_dir}/clean_passages_df.parquet')
    updated_test_df = pd.read_parquet(f'{save_dir}/updated_test_df.parquet')
    
    # Load FAISS
    faiss_index = faiss.read_index(f'{save_dir}/bioasq_faiss_index.index')
    
    with open(f'{save_dir}/passages_list.pkl', 'rb') as f:
        passages_list = pickle.load(f)
    
    with open(f'{save_dir}/passage_ids.pkl', 'rb') as f:
        passage_ids = pickle.load(f)
    
    # Load embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    # Load vectorstore
    vectorstore = FAISS.load_local(f'{save_dir}/langchain_vectorstore', embeddings)
    
    # Load config
    with open(f'{save_dir}/system_config.json', 'r') as f:
        config = json.load(f)
    
    print("✅ RAG system loaded successfully!")
    return clean_passages_df, updated_test_df, faiss_index, passages_list, passage_ids, vectorstore, config

# Usage: clean_passages_df, updated_test_df, faiss_index, passages_list, passage_ids, vectorstore, config = load_rag_system()
'''

with open(f'{save_dir}/load_system.py', 'w') as f:
    f.write(load_script)

print("✅ Deployment scripts created")

# =============================================================================
# 9. README FILE
# =============================================================================

print("9️⃣ Creating README...")

readme_content = f'''# Medical RAG Chatbot System

## Quick Setup
```bash
# 1. Install dependencies
chmod +x install.sh
./install.sh

# 2. Load system
python -c "exec(open('load_system.py').read())"
```

## System Overview
- **Query Rewriting**: Gemma-2-2B-IT
- **Embeddings**: SentenceTransformers all-MiniLM-L6-v2  
- **Vector Database**: FAISS with {len(clean_passages_df):,} medical passages
- **Answer Generation**: Llama-2-7B-chat
- **Framework**: LangChain
- **Out-of-context Filtering**: Medical keyword detection

## Performance Metrics
- **MAP**: 0.5911
- **MRR**: 0.6489
- **ROUGE-L**: 0.0640  
- **BERT-F1**: 0.8095

## Files Included
```
rag_system_checkpoint/
├── requirements.txt              # Python dependencies
├── install.sh                   # Installation script  
├── load_system.py               # Quick deployment script
├── README.md                    # This file
├── clean_passages_df.parquet    # Deduplicated medical passages
├── updated_test_df.parquet      # Test questions
├── bioasq_faiss_index.index     # FAISS vector index
├── langchain_vectorstore/       # LangChain vectorstore
├── system_config.json           # Model configurations
├── evaluation_results.json      # Performance metrics
└── sample_conversations.json    # Test cases
```

## System Requirements
- **Python**: 3.8+ 
- **RAM**: 8GB+ (16GB recommended)
- **Storage**: 20GB+ for model cache
- **GPU**: Optional but recommended
- **Internet**: Required for first-time model downloads

## Manual Installation
```bash
pip install -r requirements.txt
python -c "import nltk; nltk.download('punkt')"
```

## Usage Example
```python
# Load system
exec(open('load_system.py').read())

# Use chatbot
result = medical_chatbot("What causes diabetes?")
print(result['result'])
```

## Model Downloads (First Run Only)
- Gemma-2-2B: ~5GB
- Llama-2-7B: ~13GB  
- SentenceTransformers: ~90MB
- **Total**: ~18GB (cached automatically)

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
'''

with open(f'{save_dir}/README.md', 'w') as f:
    f.write(readme_content)

print("✅ README created")

# =============================================================================
# 8. SUMMARY
# =============================================================================

print("\n" + "="*60)
print("🎯 SAVE COMPLETE - DEPLOYMENT READY")
print("="*60)

total_size_mb = sum(os.path.getsize(os.path.join(save_dir, f)) 
                   for f in os.listdir(save_dir) if os.path.isfile(os.path.join(save_dir, f))) / (1024*1024)

print(f"📁 Save Directory: {save_dir}/")
print(f"💾 Total Size: {total_size_mb:.1f} MB")
print(f"📊 Files Saved: {len(os.listdir(save_dir))}")

print("\n🚀 FOR DEPLOYMENT:")
print("1. Download the entire folder")
print("2. Run: exec(open('load_system.py').read())")
print("3. Your RAG system will be ready!")

print("\n📋 WHAT'S SAVED:")
print("✅ requirements.txt (exact dependencies)")
print("✅ install.sh (automated setup)")
print("✅ Processed data (no need to rerun deduplication)")
print("✅ FAISS index (no need to rebuild embeddings)")  
print("✅ Model configurations (reproducible setup)")
print("✅ Evaluation results (for reporting)")
print("✅ Load scripts (quick deployment)")

print("\n⚠️  MODELS AUTO-DOWNLOAD:")
print("- Gemma, Llama, embeddings will download from HuggingFace cache")
print("- First run: ~15GB download")
print("- Subsequent runs: Fast loading from cache")

print("\n🎊 YOUR RAG SYSTEM IS DEPLOYMENT-READY!")

🔄 Saving complete RAG system for deployment...
1️⃣ Saving critical processed data...
✅ Critical data saved
2️⃣ Saving FAISS index and supporting data...
✅ FAISS and vectorstore saved
3️⃣ Saving evaluation results...
✅ Evaluation results saved
4️⃣ Saving system configuration...
✅ System configuration saved
5️⃣ Saving sample conversations...
✅ Sample conversations saved
6️⃣ Creating requirements.txt...
✅ Requirements.txt created
7️⃣ Creating installation script...
✅ Installation script created
8️⃣ Creating deployment scripts...
✅ Deployment scripts created
9️⃣ Creating README...
✅ README created

🎯 SAVE COMPLETE - DEPLOYMENT READY
📁 Save Directory: rag_system_checkpoint/
💾 Total Size: 106.0 MB
📊 Files Saved: 14

🚀 FOR DEPLOYMENT:
1. Download the entire folder
2. Run: exec(open('load_system.py').read())
3. Your RAG system will be ready!

📋 WHAT'S SAVED:
✅ requirements.txt (exact dependencies)
✅ install.sh (automated setup)
✅ Processed data (no need to rerun deduplication)
✅ FAISS index (n

In [82]:
# Task 4 Status Table - RAG with Query Rewriting using Gemma
import pandas as pd

# Create comprehensive status table for Task 4
task4_status_data = {
    'Model': [
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)',
        'RAG with Query Rewriting (Gemma + FAISS + Llama)'
    ],
    'Tasks and Comments': [
        'Data Preprocessing - 1. BioASQ dataset loading, 2. Duplicate analysis and removal, 3. Clean dataset preparation',
        'Query Rewriting Setup - 1. Gemma-2-2B-IT model loading, 2. Pipeline configuration, 3. Medical keywords and non-medical keywords',
        'Embedding & Vector Database - 1. SentenceTransformer setup, 2. FAISS index creation, 3. Vector storage',
        'LangChain Integration - 1. LangChain FAISS vectorstore, 2. HuggingFace pipeline wrapper, 3. Custom RAG chain implementation',
        'Answer Generation Setup - 1. Llama-2-7B-chat , 2. Pipeline configuration, 3. Prompt template design',
        'Out-of-Context Filtering - 1. Medical keyword detection, 2. Non-medical rejection logic, 3. Context validation system',
        'Complete RAG Pipeline - 1. Query rewriting → Retrieval → Generation, 2. End-to-end testing, 3. Error handling',
        'Evaluation Metrics - 1. MAP/MRR for retrieval, 2. ROUGE-L/BERT-F1 for generation, 3. Performance analysis',
        'System Testing - 1. Medical question validation, 2. Non-medical rejection testing, 3. Edge case handling',
        'Performance Tuning - 1. Token limits optimization, 2. Context length handling, 3. Generation parameters',
        'Deployment Preparation - 1. Model saving, 2. Configuration export, 3. Requirements documentation'
    ],
    'Status': [
        'Done',
        'Done', 
        'Done',
        'Done',
        'Done',
        'Done',
        'Done',
        'Done',
        'Done',
        'pending',
        'Done'
    ],
    'Individual Responsible': [
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Subhash',
        'Sangeeth',
        'Sangeeth'
    ],
    'Performance Metrics': [
        'Dataset: 40,221 → 27,969 passages (30% duplicates removed)',
        'Query Expansion: Medical terminology enhancement working',
        'FAISS Index: 27,969 vectors, 384 dimensions, IndexFlatIP',
        'LangChain: Vectorstore + Pipeline + RAG chain implemented',
        'Llama: 7B parameters, text generation pipeline functional',
        'Filtering: 100% accuracy on test cases (medical pass, non-medical reject)',
        'Pipeline: Query→Rewrite→Retrieve→Generate working end-to-end',
        'MAP: 0.5911, MRR: 0.6489, ROUGE-L: 0.0640, BERT-F1: 0.8095',
        'Testing: Medical Q&A working, Out-of-context rejection working',
        'Optimization: 150 max tokens, temperature 0.7, top-5 retrieval',
        'Deployment: All components saved, requirements documented'
    ]
}

# Create DataFrame
task4_status_df = pd.DataFrame(task4_status_data)

# Display the table
print("="*120)
print("TASK 4 STATUS TABLE - RAG WITH QUERY REWRITING USING GEMMA")
print("="*120)
print(task4_status_df.to_string(index=False, max_colwidth=50))
print("="*120)

print("\n🎯 TASK 4 SUMMARY:")
print(f"✅ Total Subtasks: {len(task4_status_df)}")
print(f"✅ Completed: {sum(1 for status in task4_status_df['Status'] if status == 'Done')}")
print(f"✅ Success Rate: 100%")

print("\n📊 KEY ACHIEVEMENTS:")
print("• RAG Pipeline: Gemma (query rewriting) + FAISS (retrieval) + Llama (generation)")
print("• LangChain Implementation: Full compliance with assignment requirements")
print("• Out-of-Context Filtering: Medical vs non-medical question detection")
print("• Performance: MAP 0.59, MRR 0.65, BERT-F1 0.81")
print("• Deployment Ready: Complete system saved with requirements")

print("\n🔧 TECHNICAL STACK:")
print("• Query Rewriting: google/gemma-2-2b-it")
print("• Embeddings: sentence-transformers/all-MiniLM-L6-v2")
print("• Vector DB: FAISS with 27,969 medical passages")
print("• Answer Generation: meta-llama/Llama-2-7b-chat-hf")
print("• Framework: LangChain with HuggingFace integration")

print("\n💡 NEXT STEPS RECOMMENDED:")
print("1. GPU Optimization: Implement batch processing for faster inference")
print("2. Domain Embeddings: Use medical-specific embedding models")
print("3. Conversation Memory: Add chat history for multi-turn conversations")
print("4. Fine-tuning: Adapt models on domain-specific medical data")
print("5. Deployment: Scale with Docker containers and load balancing")

# Save the status table
task4_status_df.to_csv('task4_status_report.csv', index=False)
print(f"\n📁 Status table saved as: task4_status_report.csv")

TASK 4 STATUS TABLE - RAG WITH QUERY REWRITING USING GEMMA
                                           Model                                 Tasks and Comments  Status Individual Responsible                                Performance Metrics
RAG with Query Rewriting (Gemma + FAISS + Llama) Data Preprocessing - 1. BioASQ dataset loading,...    Done                Subhash Dataset: 40,221 → 27,969 passages (30% duplicat...
RAG with Query Rewriting (Gemma + FAISS + Llama) Query Rewriting Setup - 1. Gemma-2-2B-IT model ...    Done                Subhash Query Expansion: Medical terminology enhancemen...
RAG with Query Rewriting (Gemma + FAISS + Llama) Embedding & Vector Database - 1. SentenceTransf...    Done                Subhash FAISS Index: 27,969 vectors, 384 dimensions, In...
RAG with Query Rewriting (Gemma + FAISS + Llama) LangChain Integration - 1. LangChain FAISS vect...    Done                Subhash LangChain: Vectorstore + Pipeline + RAG chain i...
RAG with Query Rewriting (Gemma

In [84]:
pd.read_csv("task4_status_report.csv")

,Model,Tasks and Comments,Status,Individual Responsible,Performance Metrics
0,RAG with Query Rewriting (Gemma + FAISS + Llama),"Data Preprocessing - 1. BioASQ dataset loading, 2. Duplicate analysis and removal, 3. Clean dataset preparation",Done,Subhash,"Dataset: 40,221 → 27,969 passages (30% duplicates removed)"
1,RAG with Query Rewriting (Gemma + FAISS + Llama),"Query Rewriting Setup - 1. Gemma-2-2B-IT model loading, 2. Pipeline configuration, 3. Medical keywords and non-medical keywords",Done,Subhash,Query Expansion: Medical terminology enhancement working
2,RAG with Query Rewriting (Gemma + FAISS + Llama),"Embedding & Vector Database - 1. SentenceTransformer setup, 2. FAISS index creation, 3. Vector storage",Done,Subhash,"FAISS Index: 27,969 vectors, 384 dimensions, IndexFlatIP"
3,RAG with Query Rewriting (Gemma + FAISS + Llama),"LangChain Integration - 1. LangChain FAISS vectorstore, 2. HuggingFace pipeline wrapper, 3. Custom RAG chain implementation",Done,Subhash,LangChain: Vectorstore + Pipeline + RAG chain implemented
4,RAG with Query Rewriting (Gemma + FAISS + Llama),"Answer Generation Setup - 1. Llama-2-7B-chat , 2. Pipeline configuration, 3. Prompt template design",Done,Subhash,"Llama: 7B parameters, text generation pipeline functional"
5,RAG with Query Rewriting (Gemma + FAISS + Llama),"Out-of-Context Filtering - 1. Medical keyword detection, 2. Non-medical rejection logic, 3. Context validation system",Done,Subhash,"Filtering: 100% accuracy on test cases (medical pass, non-medical reject)"
6,RAG with Query Rewriting (Gemma + FAISS + Llama),"Complete RAG Pipeline - 1. Query rewriting → Retrieval → Generation, 2. End-to-end testing, 3. Error handling",Done,Subhash,Pipeline: Query→Rewrite→Retrieve→Generate working end-to-end
7,RAG with Query Rewriting (Gemma + FAISS + Llama),"Evaluation Metrics - 1. MAP/MRR for retrieval, 2. ROUGE-L/BERT-F1 for generation, 3. Performance analysis",Done,Subhash,"MAP: 0.5911, MRR: 0.6489, ROUGE-L: 0.0640, BERT-F1: 0.8095"
8,RAG with Query Rewriting (Gemma + FAISS + Llama),"System Testing - 1. Medical question validation, 2. Non-medical rejection testing, 3. Edge case handling",Done,Subhash,"Testing: Medical Q&A working, Out-of-context rejection working"
9,RAG with Query Rewriting (Gemma + FAISS + Llama),"Performance Tuning - 1. Token limits optimization, 2. Context length handling, 3. Generation parameters",pending,Sangeeth,"Optimization: 150 max tokens, temperature 0.7, top-5 retrieval"
